# Minimal Surfaces Demo

This notebook reproduces the key figures from `minsurf`:
1. Catenoid relaxation (area descent, neck convergence)
2. Associate-family morph (catenoid → helicoid, 6 frames)
3. Enneper surface
4. Stability threshold sweep

## Mathematical background

A **minimal surface** has zero mean curvature $H = 0$ everywhere.
In the discrete setting this means $\mathbf{L}\mathbf{x} = 0$, where
$\mathbf{L}$ is the cotangent Laplacian.

We find minimal surfaces by **implicit mean-curvature flow**:
$$
(\mathbf{M} + \tau \mathbf{L})\,\mathbf{x}^{n+1} = \mathbf{M}\,\mathbf{x}^n
$$
which monotonically decreases area and converges to $\mathbf{L}\mathbf{x}^* = 0$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
%matplotlib inline
plt.rcParams['figure.dpi'] = 100

from minsurf import boundaries, exact, flow, metrics, operators, visualize

## 1. Catenoid Relaxation

Seed: straight cylinder between two coaxial rings (separation=1, radius=1).  
Exact: catenoid with neck $a \approx 0.848$ solving $R = a\cosh(h/a)$.

In [ ]:
# Build seed and solve
seed = boundaries.two_rings(separation=1.0, radius=1.0, n_z=24, n_theta=48)
print(f"Seed: {seed.n_vertices()} vertices, area = {seed.total_area():.4f}")

mesh, hist = flow.solve(seed, method='implicit', tau=0.05, max_iter=500, tol=1e-6)
print(f"Solved: {len(hist.area)} iters, area = {mesh.total_area():.4f}, "
      f"max|H| = {metrics.max_mean_curvature(mesh):.2e}, converged = {hist.converged}")

In [ ]:
# Exact catenoid comparison
exact_mesh, a_exact = exact.catenoid_for_rings(separation=1.0, radius=1.0)
l2_err = metrics.l2_error_to_catenoid(mesh, a_exact)
neck = metrics.neck_radius(mesh)
print(f"Exact neck a = {a_exact:.4f}")
print(f"Measured neck = {neck:.4f}  (error {100*abs(neck-a_exact)/a_exact:.2f}%)")
print(f"L2 radial error = {l2_err:.4e}")

In [ ]:
# Convergence history
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
iters = range(len(hist.area))
ax1.plot(iters, hist.area, 'b-', lw=1.5)
ax1.set(xlabel='Iteration', ylabel='Total area', title='Area descent')
ax1.grid(alpha=0.3)
ax2.semilogy(iters, hist.residual, 'r-', lw=1.5)
ax2.set(xlabel='Iteration', ylabel='max |H|', title='Residual convergence')
ax2.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Profile: computed vs exact
visualize.render_catenoid_profile(mesh, a_exact, show=True)

In [ ]:
# 3-D render coloured by |H|
visualize.render_surface(mesh, title='Catenoid (computed)', show=True)

## 2. Associate Family: Catenoid ↔ Helicoid

The W-E associate family $f \mapsto e^{it}f$ deforms the catenoid ($t=0$)
isometrically into the helicoid ($t=\pi/2$).
The metric $E = G = \cosh^2 u$, $F = 0$ is preserved for all $t$.

In [ ]:
t_values = np.linspace(0, np.pi/2, 6)
labels = ['Catenoid (t=0)', 't=π/10', 't=π/5', 't=3π/10', 't=2π/5', 'Helicoid (t=π/2)']

fig = plt.figure(figsize=(15, 4))
for k, (t, label) in enumerate(zip(t_values, labels)):
    fam = exact.associate_family(t=t, a=1.0, separation=2.0, n_theta=32, n_z=20)
    ax = fig.add_subplot(1, 6, k+1, projection='3d')
    V, F = fam.V, fam.F
    polys = [[V[F[i,0]], V[F[i,1]], V[F[i,2]]] for i in range(min(F.shape[0], 2000))]
    ax.add_collection3d(Poly3DCollection(polys, facecolors='#4dd9d9', edgecolors='none', alpha=0.7))
    # Set limits
    lim = 1.5
    ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim); ax.set_zlim(-1.5, 1.5)
    ax.set_title(label, fontsize=8)
    ax.axis('off')
plt.suptitle('Associate Family: Catenoid → Helicoid (isometric deformation)', y=1.02)
plt.tight_layout()
plt.show()

## 3. Enneper Surface

$$X(u,v) = \bigl(u - u^3/3 + uv^2,\; v - v^3/3 + u^2v,\; u^2 - v^2\bigr)$$

A complete minimal surface with self-intersections for $|u|, |v| > 1$.

In [ ]:
enn = exact.enneper(r_max=0.8, n_u=40, n_v=40)
print(f"Enneper: {enn.n_vertices()} vertices, area = {enn.total_area():.4f}")
H_enn = operators.mean_curvature(enn)
print(f"Interior max|H| = {H_enn[~enn.boundary].max():.4e}  (should be ≈ 0)")
visualize.render_surface(enn, title='Enneper Surface (exact)', show=True)

## 4. Stability Threshold Sweep

The catenoid between equal coaxial rings exists only while separation/radius ≤ 1.3255.
Beyond this, the flow collapses toward the Goldschmidt two-disk configuration.

In [ ]:
from minsurf import stability

separations = np.linspace(0.5, 1.8, 8)
necks, ratios, regimes = [], [], []

for sep in separations:
    seed_s = boundaries.two_rings(separation=sep, radius=1.0, n_z=16, n_theta=32)
    solved_s, _ = flow.solve(seed_s, method='implicit', tau=0.05, max_iter=300, tol=1e-4)
    info = stability.analyze(solved_s)
    neck_s = metrics.neck_radius(solved_s)
    necks.append(neck_s)
    ratios.append(info['ratio'])
    regimes.append(info['regime'])
    print(f"  sep={sep:.2f}  ratio={info['ratio']:.3f}  neck={neck_s:.3f}  regime={info['regime']}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
colors = ['#4dd9d9' if r == 'stable' else '#f04050' for r in regimes]
ax.scatter(separations, necks, c=colors, s=80, zorder=5)
ax.axvline(stability.CRITICAL_RATIO, color='orange', lw=1.5, ls='--',
           label=f'Critical ratio = {stability.CRITICAL_RATIO}')
ax.set(xlabel='Separation / Radius', ylabel='Neck radius',
       title='Catenoid Stability: Neck Radius vs Separation')
ax.legend()
ax.grid(alpha=0.3)
# Legend patches
from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(color='#4dd9d9', label='stable'),
    Patch(color='#f04050', label='collapsed'),
    plt.Line2D([0],[0], color='orange', ls='--', label=f'threshold={stability.CRITICAL_RATIO}'),
])
plt.tight_layout()
plt.show()

## 5. Gauss–Bonnet Check

For a sphere: $\sum_i K_i A_i \approx 4\pi \approx 12.566$.

In [ ]:
from minsurf.metrics import gauss_bonnet_residual
from minsurf.mesh import Mesh

# Build a UV-sphere
def make_sphere(n=20):
    thetas = np.linspace(0, np.pi, n+2)[1:-1]
    phis = np.linspace(0, 2*np.pi, 2*n, endpoint=False)
    V = [[0,0,1]]
    for th in thetas:
        for ph in phis:
            V.append([np.sin(th)*np.cos(ph), np.sin(th)*np.sin(ph), np.cos(th)])
    V.append([0,0,-1])
    V = np.array(V)
    N = len(phis); F = []
    gi = lambda i, j: 1 + i*N + (j % N)
    for j in range(N): F.append([0, gi(0,j), gi(0,j+1)])
    for i in range(n-1):
        for j in range(N):
            F += [[gi(i,j), gi(i,j+1), gi(i+1,j)], [gi(i,j+1), gi(i+1,j+1), gi(i+1,j)]]
    for j in range(N): F.append([gi(n-1,j+1), gi(n-1,j), len(V)-1])
    return Mesh(V=V, F=np.array(F, dtype=np.int64), boundary=np.zeros(len(V), dtype=bool))

sphere = make_sphere(20)
gb_err = gauss_bonnet_residual(sphere)
K = operators.gaussian_curvature(sphere)
A = operators.vertex_areas(sphere)
print(f"∑ K_i A_i = {np.sum(K*A):.4f}  (expected 4π = {4*np.pi:.4f})")
print(f"Relative Gauss–Bonnet error: {gb_err:.3%}")